# FERRUS OSSEUS — Fregate 04
## FLOTTE FERRUS | AD MAJOREM GLORIAM IMPERATORIS

**Pipeline** : Mesh T-pose (sans rig) → FBX rigé (R15 | Mixamo | DeepMotion)

```
IN/      ← avatar brut .glb / .gltf / .obj / .fbx (T-pose, SANS squelette)
OUT/     ← avatar_rigged_*.fbx + rapport_osseus.json
```

---
**Prerequis** : avatar en T-pose (bras horizontaux, jambes droites).
Aucun squelette existant requis.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 1 — CONFIGURATION
# Editer ici avant de lancer
# ══════════════════════════════════════════════════════════════

# ── Fichier source ────────────────────────────────────────────
INPUT_FILE = "avatar_P1.glb"        # Nom du fichier dans IN/
                                     # Formats : .glb .gltf .obj .fbx

# ── Template squelette ───────────────────────────────────────
TEMPLATE = "r15"                     # r15 | mixamo | deepmotion
#   r15         : 15 bones Roblox R15 (pour FERRUS ANIMUS / Roblox)
#   mixamo      : 26 bones Mixamo (pour Adobe Mixamo / unity)
#   deepmotion  : 52 bones DeepMotion avec doigts (pour FERRUS ANIMUS)

# ── Dossiers (ne pas modifier sauf cas particulier) ───────────
IN_DIR    = "/content/04_FERRUS_OSSEUS/IN"
OUT_DIR   = "/content/04_FERRUS_OSSEUS/OUT"
CODE_DIR  = "/content/04_FERRUS_OSSEUS/codebase"

# ── Repo GitHub ───────────────────────────────────────────────
GITHUB_REPO  = "kioka8877-ux/FLOTTE-FERRUS"
GITHUB_TOKEN = ""  # Laisser vide si repo public ou si deja clone

print("[OSSEUS] Configuration :")
print(f"  Input    : {INPUT_FILE}")
print(f"  Template : {TEMPLATE}")
print(f"  IN_DIR   : {IN_DIR}")
print(f"  OUT_DIR  : {OUT_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 2 — INSTALLATION BLENDER
# ══════════════════════════════════════════════════════════════

import subprocess, os, sys

def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout[-2000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        raise RuntimeError(f"Commande echouee: {cmd}")
    return result

# Verifier si Blender est deja installe
blender_check = subprocess.run("blender --version", shell=True, capture_output=True, text=True)

if blender_check.returncode != 0:
    print("[OSSEUS] Installation Blender...")
    run("apt-get install -y blender > /dev/null 2>&1 || snap install blender --classic")
else:
    ver = blender_check.stdout.strip().split("\n")[0]
    print(f"[OSSEUS] Blender deja installe : {ver}")

run("blender --version")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 3 — CLONE DU REPO ET MONTAGE DRIVE
# ══════════════════════════════════════════════════════════════

import os

# Optionnel : monter Google Drive pour persistance des fichiers
MOUNT_DRIVE = False  # Passer a True pour utiliser votre Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Clone du repo
if not os.path.exists("/content/04_FERRUS_OSSEUS"):
    print("[OSSEUS] Clone du repo...")
    if GITHUB_TOKEN:
        run(f"git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git /content/FLOTTE-FERRUS")
    else:
        run(f"git clone https://github.com/{GITHUB_REPO}.git /content/FLOTTE-FERRUS")
    
    # Creer un symlink vers la fregate
    run("ln -sf /content/FLOTTE-FERRUS/04_FERRUS_OSSEUS /content/04_FERRUS_OSSEUS")
else:
    print("[OSSEUS] Dossier 04_FERRUS_OSSEUS deja present")
    run("cd /content/FLOTTE-FERRUS && git pull")

# Creer les dossiers IN et OUT si absents
os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"[OSSEUS] Structure prete")
run(f"ls -la {IN_DIR}/")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 4 — UPLOAD AVATAR (si pas deja dans IN/)
# ══════════════════════════════════════════════════════════════

import os

input_full_path = os.path.join(IN_DIR, INPUT_FILE)

if not os.path.exists(input_full_path):
    print(f"[OSSEUS] Fichier {INPUT_FILE} non trouve dans IN/")
    print("[OSSEUS] Upload manuel via le panneau Files de Colab,")
    print(f"[OSSEUS] ou deposer dans {IN_DIR}/")
    
    # Upload interactif
    from google.colab import files
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join(IN_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(data)
        print(f"[OSSEUS] Upload OK : {dest}")
        INPUT_FILE = fname  # Mettre a jour si nom different
        input_full_path = dest
else:
    size_mb = os.path.getsize(input_full_path) / (1024*1024)
    print(f"[OSSEUS] Fichier trouve : {input_full_path} ({size_mb:.2f} Mo)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 5 — PIPELINE OSSEUS
# Execution du rigging automatique via Blender headless
# ══════════════════════════════════════════════════════════════

import os, subprocess, json
from pathlib import Path

stem = Path(INPUT_FILE).stem
output_file    = f"avatar_rigged_{stem}_{TEMPLATE}.fbx"
output_path    = os.path.join(OUT_DIR, output_file)
report_path    = os.path.join(OUT_DIR, f"rapport_osseus_{stem}.json")
script_path    = os.path.join(CODE_DIR, "osseus_core.py")
input_full_path = os.path.join(IN_DIR, INPUT_FILE)

print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"[OSSEUS] Lancement pipeline OSSEUS")
print(f"[OSSEUS]   Input    : {input_full_path}")
print(f"[OSSEUS]   Template : {TEMPLATE}")
print(f"[OSSEUS]   Output   : {output_path}")
print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

cmd = (
    f"blender --background --python {script_path} -- "
    f"--input   '{input_full_path}' "
    f"--output  '{output_path}' "
    f"--template {TEMPLATE} "
    f"--report  '{report_path}'"
)

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

# Afficher les logs OSSEUS seulement
all_output = result.stdout + result.stderr
osseus_lines = [l for l in all_output.split("\n") if l.startswith("[OSSEUS]")]
print("\n".join(osseus_lines))

if result.returncode != 0 and not os.path.exists(output_path):
    print("\n[OSSEUS] ERREUR — Logs complets :")
    print(all_output[-3000:])
    raise RuntimeError("Pipeline OSSEUS echoue")

print(f"\n[OSSEUS] Sortie : {output_path}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 6 — RAPPORT ET VALIDATION
# ══════════════════════════════════════════════════════════════

import json, os

if os.path.exists(report_path):
    with open(report_path) as f:
        rapport = json.load(f)
    
    status = rapport.get('status', 'UNKNOWN')
    icon   = '✓' if status == 'SUCCESS' else '✗'
    
    print(f"""[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[OSSEUS] RAPPORT OSSEUS {icon} {status}
[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Input         : {rapport.get('input', '-')}
  Template      : {rapport.get('template', '-')}
  Vertices      : {rapport.get('vertices', '-'):,}
  Faces         : {rapport.get('faces', '-'):,}
  Hauteur mesh  : {rapport.get('height', '-')}
  Largeur mesh  : {rapport.get('width', '-')}
  Bones crees   : {rapport.get('bones_count', '-')}
  Auto Weights  : {rapport.get('auto_weights', '-')}
  Taille sortie : {rapport.get('output_size_mb', '-')} Mo
""")
    
    if status == 'ERROR':
        print(f"  ERREUR : {rapport.get('error', 'inconnue')}")
else:
    print("[OSSEUS] Rapport JSON introuvable")

# Verifier que le fichier FBX existe
if os.path.exists(output_path):
    size = os.path.getsize(output_path) / (1024*1024)
    print(f"[OSSEUS] FBX rige pret : {output_path} ({size:.2f} Mo)")
    print(f"[OSSEUS] Deposer dans FERRUS ANIMUS IN/ pour retargeting")
else:
    print(f"[OSSEUS] ERREUR : FBX non genere")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 7 — DOWNLOAD SORTIE
# ══════════════════════════════════════════════════════════════

from google.colab import files
import os

if os.path.exists(output_path):
    print(f"[OSSEUS] Download : {output_file}")
    files.download(output_path)

if os.path.exists(report_path):
    report_file = os.path.basename(report_path)
    print(f"[OSSEUS] Download rapport : {report_file}")
    files.download(report_path)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 8 — PUSH VERS GITHUB (optionnel)
# ══════════════════════════════════════════════════════════════

import subprocess, os

PUSH_TO_GITHUB = False  # Passer a True pour auto-push

if PUSH_TO_GITHUB and GITHUB_TOKEN:
    target_in_repo = f"04_FERRUS_OSSEUS/OUT/{output_file}"
    
    run(f"cp '{output_path}' /content/FLOTTE-FERRUS/{target_in_repo}")
    
    os.chdir("/content/FLOTTE-FERRUS")
    run(f"git config user.email 'osseus@flotte-ferrus.ai'")
    run(f"git config user.name 'FERRUS OSSEUS'")
    run(f"git add {target_in_repo}")
    run(f"git commit -m '[OSSEUS] Add {output_file} (template={TEMPLATE})'")
    run(f"git push https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git main")
    print(f"[OSSEUS] Push GitHub OK : {target_in_repo}")
else:
    print("[OSSEUS] Push GitHub desactive (PUSH_TO_GITHUB=False)")